<a id="load-models"></a>
# Évaluation "à froid" des modèles existants sur CorBaMa

On charge les modèles déjà entraînés (module 06 : mot-entier, module 07a : SentencePiece broadcasting) et on les évalue directement sur corbama_corpus.tsv, sans aucun réentraînement — c'est le test le plus honnête fait sur ce projet, car CorBaMa n'a aucun lien avec le lexique ni la règle du module 04.

In [7]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from bambara_pos_utils import *

import torch
from sklearn.metrics import accuracy_score, classification_report
import sentencepiece as spm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", device)

corbama_data = load_bambara_corpus(Path.cwd() / "corbama_corpus.tsv")
print(f"Corpus réel CorBaMa : {len(corbama_data)} phrases")

Device : cpu
Corpus chargé : 166361 phrases.
Corpus réel CorBaMa : 166361 phrases


<a id="word-level-eval"></a>
# Modèle mot-entier (module 06)

In [8]:
model_06, word_to_ix_06, hp_06 = load_checkpoint(
    Path.cwd().parent / "06_final_training_and_inference" / "models" / "bambara_pos_best.pth",
    device
)

def evaluate_word_model_on_corpus(model, word_to_ix, sentences, device):
    model.eval()
    all_true, all_pred = [], []
    with torch.no_grad():
        for words, tags in sentences:
            ids = [word_to_ix.get(w, word_to_ix["<UNK>"]) for w in words]
            input_tensor = torch.tensor([ids], dtype=torch.long).to(device)
            preds = torch.argmax(model(input_tensor), dim=-1).squeeze(0).cpu().tolist()
            all_true.extend(tags)
            all_pred.extend([id2label[p] for p in preds])
    return all_true, all_pred

true_06, pred_06 = evaluate_word_model_on_corpus(model_06, word_to_ix_06, corbama_data, device)
acc_06 = accuracy_score(true_06, pred_06)
print(f"Accuracy module 06 sur CorBaMa (réel) : {acc_06*100:.2f}%\n")
print(classification_report(true_06, pred_06, zero_division=0))

Accuracy module 06 sur CorBaMa (réel) : 64.16%

              precision    recall  f1-score   support

         ADJ       0.99      0.10      0.19     80453
         ADV       0.39      0.19      0.26     23045
         AUX       0.64      0.58      0.61    341048
        CONJ       0.61      0.54      0.57    101866
         DET       0.40      0.49      0.44     79879
         NOM       0.49      0.79      0.61    635374
        PART       0.45      0.24      0.31     55415
       POSTP       0.87      0.46      0.61    240354
        PRON       0.91      0.74      0.82    361139
       PUNCT       1.00      0.91      0.96    389376
       VERBE       0.47      0.41      0.43    323993

    accuracy                           0.64   2631942
   macro avg       0.66      0.50      0.53   2631942
weighted avg       0.69      0.64      0.64   2631942



In [9]:
# Taux de mots inconnus (<UNK>) — indicateur clé pour motiver l'amélioration de la tokenisation
unk_count = sum(1 for words, _ in corbama_data for w in words if w not in word_to_ix_06)
total_tokens = sum(len(words) for words, _ in corbama_data)
print(f"Taux de mots inconnus (vocabulaire mot-entier) : {100*unk_count/total_tokens:.1f}%")
print(f"({unk_count} mots sur {total_tokens} ne sont pas dans le vocabulaire appris sur Bayelemabaga)")

Taux de mots inconnus (vocabulaire mot-entier) : 18.8%
(495438 mots sur 2631942 ne sont pas dans le vocabulaire appris sur Bayelemabaga)


<a id="subword-level-eval"></a>
# Modèle sous-mots (module 07a, broadcasting)

In [12]:
sp = spm.SentencePieceProcessor(
    model_file=str(Path.cwd().parent / "07_sentencepiece_tokenization" / "bambara_sp.model")
)

checkpoint_path = (
    Path.cwd().parent
    / "07_sentencepiece_tokenization"
    / "07a_broadcast_scheme"
    / "models"
    / "bambara_pos_sp_broadcast.pth"
)

checkpoint = torch.load(checkpoint_path, map_location=device)
state_dict = checkpoint.get("model_state_dict", checkpoint.get("state_dict", checkpoint))

embedding_weight = state_dict["word_embeddings.weight"]
output_weight = state_dict["hidden2tag.weight"]

model_07 = LSTMTaggerWithBatch(
    embedding_weight.shape[1],
    output_weight.shape[1] // 2,
    embedding_weight.shape[0],
    output_weight.shape[0],
).to(device)

model_07.load_state_dict(state_dict)
model_07.eval()

hp_07 = checkpoint.get("hyperparameters", checkpoint.get("hp", {}))

acc_07, report_07 = evaluate_subword_model_word_level(
    model_07, corbama_data, sp, id2label, "broadcast", device
)
print(f"Accuracy module 07a sur CorBaMa (réel) : {acc_07*100:.2f}%\n")
print(report_07)

Accuracy module 07a sur CorBaMa (réel) : 65.60%

              precision    recall  f1-score   support

         ADJ       0.99      0.10      0.19     80453
         ADV       0.40      0.19      0.26     23045
         AUX       0.65      0.65      0.65    341048
        CONJ       0.61      0.54      0.57    101866
         DET       0.40      0.49      0.44     79879
         NOM       0.51      0.80      0.63    635374
        PART       0.45      0.24      0.31     55415
       POSTP       0.87      0.47      0.61    240354
        PRON       0.90      0.75      0.82    361139
       PUNCT       0.96      0.94      0.95    389376
       VERBE       0.50      0.39      0.44    323993

    accuracy                           0.66   2631942
   macro avg       0.66      0.51      0.53   2631942
weighted avg       0.69      0.66      0.65   2631942



In [13]:
# Taux de fragmentation moyen (sous-tokens par mot) sur CorBaMa
def avg_pieces_per_word(sentences, sp):
    total_pieces, total_words = 0, 0
    for words, _ in sentences:
        for w in words:
            total_pieces += len(sp.encode(w, out_type=str))
            total_words += 1
    return total_pieces / total_words

frag = avg_pieces_per_word(corbama_data, sp)
print(f"Sous-tokens par mot en moyenne sur CorBaMa : {frag:.2f}")

Sous-tokens par mot en moyenne sur CorBaMa : 1.42


<a id="comparison"></a>
# Comparaison finale : mot-entier vs sous-mots, sur données réelles

In [15]:
print(f"{'Modèle':<35}{'Accuracy CorBaMa':<20}")
print(f"{'Mot-entier (module 06)':<35}{acc_06*100:.2f}%")
print(f"{'Sous-mots broadcasting (module 07a) : ':<35}{acc_07*100:.2f}%")
print(f"\nTaux de mots inconnus (mot-entier) : {100*unk_count/total_tokens:.1f}%")
print(f"Sous-tokens par mot en moyenne (sous-mots) : {frag:.2f}")

Modèle                             Accuracy CorBaMa    
Mot-entier (module 06)             64.16%
Sous-mots broadcasting (module 07a) : 65.60%

Taux de mots inconnus (mot-entier) : 18.8%
Sous-tokens par mot en moyenne (sous-mots) : 1.42


## Interprétation 

Trois chiffres à présenter ensemble : l'accuracy des deux approches sur données réelles (plus basse que sur notre propre test set — c'est attendu et normal, pas un échec), le taux de <UNK> du modèle mot-entier (montre concrètement le problème que SentencePiece est censé résoudre), et la fragmentation moyenne (indicateur direct de "combien le tokeniseur découpe"). Ces résultats motivent objectivement le travail du module 09.